[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/02_%E4%BB%AE%E8%AA%AC%E6%A4%9C%E5%AE%9A.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計2級-02. 仮説検定

「偶然では説明しにくい差か？」を確率で判断する方法を学びます。
統計検定2級の中心テーマです。

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
rng = np.random.default_rng(0)   # 乱数生成器（seedで結果を固定）

## 1. 検定の考え方（5ステップ）

1. **帰無仮説 H₀**：差はない（例：平均は200である）
2. **対立仮説 H₁**：差がある（例：平均は200でない）
3. **有意水準 α** を決める（ふつう 0.05）
4. データから **検定統計量** と **p値** を計算
5. **p値 < α なら H₀ を棄却**（＝差があると判断）

> p値 = 「H₀が正しいと仮定したとき、観測されたような差が偶然起こる確率」

## 2. 1標本t検定（母平均の検定）

ある工場のお菓子は「内容量200g」とうたっている。
標本を測ったら本当に200gといえるか？

In [ ]:
data = np.array([198, 203, 197, 205, 199, 201, 196, 204, 200, 197,   # 内容量の標本
                 195, 199, 198, 200, 196])
t_stat, p = stats.ttest_1samp(data, popmean=200)  # 母平均=200 との1標本t検定
print(f'標本平均: {data.mean():.2f}')
print(f't統計量: {t_stat:.3f},  p値: {p:.3f}')
alpha = 0.05                    # 有意水準
print('結論:', '200gと言えない（棄却）' if p < alpha else '200gでないとは言えない')

## 3. 2標本t検定（2グループの平均の差）

新しい教え方(B)は従来(A)より点数が高いか？ 2クラスを比較します。

In [ ]:
classA = rng.normal(62, 12, 30)            # A組（平均62）
classB = rng.normal(70, 12, 30)            # B組（平均70）
t_stat, p = stats.ttest_ind(classA, classB)  # 2標本t検定（平均の差）
print(f'Aの平均 {classA.mean():.1f}, Bの平均 {classB.mean():.1f}')
print(f't={t_stat:.3f}, p={p:.4f}')
print('結論:', '差は有意' if p < 0.05 else '差は有意でない')

## 4. 母比率の検定（A/Bテストの前ふり）

サイコロを600回ふって1の目が80回。「いかさま（出やすい）」と言えるか？
理論値は 600 × 1/6 = 100 回。

In [ ]:
from statsmodels.stats.proportion import proportions_ztest  # 母比率の検定
count, nobs = 80, 600           # 1の目の回数と総試行
z, p = proportions_ztest(count, nobs, value=1/6, alternative='two-sided')  # 比率=1/6の検定
print(f'観測比率 {count/nobs:.3f} (理論 {1/6:.3f})')
print(f'z={z:.3f}, p={p:.4f}')
print('結論:', 'いかさまの疑い' if p < 0.05 else '偶然の範囲')

> 📝 **第1種・第2種の誤り**
> - 第1種の誤り：本当は差がないのに「ある」とする（確率=α）
> - 第2種の誤り：本当は差があるのに「ない」とする（確率=β）

---
## 🏆 練習問題

**問1.** ある飲料の糖度は「平均10度」とされる。標本
`[10.5,9.8,10.2,11.0,10.7,10.1,10.9,10.4]` で、10度といえるか1標本t検定で調べよう。

**問2.** `sales_btob.csv` で「展示会」と「Web広告」の商談金額の平均に差があるか、
2標本t検定で調べよう。

**問3.** コインを100回投げて表が60回。「ゆがんだコイン」と言えるか母比率の検定で調べよう。

In [ ]:
# 問1


In [ ]:
# 問2
btob = pd.read_csv('../data/sales_btob.csv')   # 商談データを読み込む

### 📋 使うデータ：`sales_btob.csv`（架空のBtoB商談400件）
1行＝商談1件。`受注`は 1=成約 / 0=失注。

| 商談ID | 受注日 | 業界 | 獲得チャネル | 商談金額 | 担当者 | 受注 |
|---|---|---|---|---|---|---|
| D0001 | 2026-01-15 | 小売 | 紹介 | 1,211,000 | 佐藤 | 0 |
| D0002 | 2026-02-05 | 医療 | テレアポ | 400,000 | 田中 | 0 |
| D0003 | 2026-02-19 | IT | 紹介 | 542,000 | 高橋 | 1 |

In [ ]:
# 問3


> 🔑 **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから 👉 **[02_仮説検定 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/02_%E4%BB%AE%E8%AA%AC%E6%A4%9C%E5%AE%9A.md)**